# Лабораторная работа №7
## Первая нейронная сеть: бинарные входы, обучение и интерпретация

В работе используется маленькая задача, в которой числа **0–15** представлены четырьмя битами.  
Нейронная сеть должна определить одну из **4 групп**, кодируемых двумя выходными битами.

Учебная цель — понять, что происходит между `model.fit()` и `model.predict()`, а не добиться сложной архитектуры.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

np.random.seed(42)
tf.random.set_seed(42)


## 1. Как число превращается во вход нейросети
Например, число 5 в 4-битном виде — `0101`.

В нашем учебном примере группа определяется остатком от деления на 4:
- группа 0 → `00`
- группа 1 → `01`
- группа 2 → `10`
- группа 3 → `11`

То есть сеть учится восстанавливать двухбитный код группы.


In [ ]:
def to_bits(number, width):
    return [int(bit) for bit in format(number, f"0{width}b")]

for n in range(8):
    group = n % 4
    print(n, "->", to_bits(n, 4), "| группа", group, "->", to_bits(group, 2))


### Задание 1
Объясните своими словами:
1. почему у сети 4 входа;
2. почему достаточно 2 выходов для четырёх групп;
3. что означает выход `10`.


## 2. Формирование обучающих данных

In [ ]:
X = []
y = []

for number in range(16):
    X.append(to_bits(number, 4))
    group = number % 4
    y.append(to_bits(group, 2))

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.float32)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nПервые строки:")
for i in range(8):
    print(i, X[i].astype(int), "->", y[i].astype(int))


### Задание 2
Найдите строку для числа 13. Какой бинарный вход получает сеть? Какой правильный выход ей соответствует?


## 3. Архитектура
Используем один слой `Dense(2)` без скрытых слоёв. Каждый из двух выходных нейронов получает четыре входных значения, умножает их на веса, добавляет смещение и применяет sigmoid.


In [ ]:
model = keras.Sequential([
    keras.Input(shape=(4,)),
    keras.layers.Dense(2, activation="sigmoid")
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.05),
    loss="binary_crossentropy",
    metrics=[keras.metrics.BinaryAccuracy(name="bit_accuracy")]
)

model.summary()


### Задание 3
По `model.summary()` определите число обучаемых параметров. Проверьте расчёт вручную:  
`4×2` весов + `2` смещения.


## 4. Обучение

In [ ]:
history = model.fit(
    X, y,
    epochs=200,
    batch_size=4,
    verbose=0
)

print("Последний loss:", history.history["loss"][-1])
print("Последняя точность по битам:", history.history["bit_accuracy"][-1])


### Задание 4
Что такое эпоха? Что меняется в модели от эпохи к эпохе? Почему уменьшение `loss` обычно является хорошим признаком?


## 5. Кривые обучения

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"])
plt.xlabel("Эпоха")
plt.ylabel("Loss")
plt.title("Изменение функции потерь")
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history.history["bit_accuracy"])
plt.xlabel("Эпоха")
plt.ylabel("Точность по выходным битам")
plt.title("Изменение точности")
plt.show()


### Задание 5
Опишите поведение двух графиков. Примерно после какой эпохи изменение становится небольшим?


## 6. Предсказания и порог 0.5

In [ ]:
prob = model.predict(X, verbose=0)
pred_bits = (prob >= 0.5).astype(int)

for number in range(16):
    print(
        f"{number:2d} | вход {X[number].astype(int)} | "
        f"выход сети {np.round(prob[number], 3)} | "
        f"после порога {pred_bits[number]} | "
        f"правильно {y[number].astype(int)}"
    )


### Задание 6
Почему выход sigmoid — это не сразу 0 или 1? Что делает порог `0.5`?  
Сравните вывод сети для одного уверенного и одного менее уверенного примера.


## 7. Точность группы целиком

In [ ]:
exact_matches = np.all(pred_bits == y.astype(int), axis=1)
exact_accuracy = exact_matches.mean()

print("Точная доля правильно определённых групп:", exact_accuracy)
print("Ошибочные числа:", np.where(~exact_matches)[0].tolist())


### Задание 7
Чем «точность по битам» отличается от «правильно определённой группы целиком»? Почему для нашей задачи второй показатель понятнее?


## 8. Веса и смещения

In [ ]:
weights, biases = model.layers[0].get_weights()

print("Матрица весов:")
print(np.round(weights, 3))
print("\nСмещения:")
print(np.round(biases, 3))


### Задание 8
Определите, какие входные биты сильнее всего влияют на первый и второй выход.  
Не пытайтесь трактовать знак веса отдельно от остальных весов и смещения: нейрон использует их совместно.


## 9. Ручная проверка одного нейрона

In [ ]:
number = 5
x = X[number]

z = x @ weights + biases
manual_sigmoid = 1 / (1 + np.exp(-z))

print("Число:", number)
print("Вход:", x.astype(int))
print("z =", np.round(z, 4))
print("sigmoid(z) =", np.round(manual_sigmoid, 4))
print("model.predict =", np.round(model.predict(x.reshape(1, -1), verbose=0)[0], 4))


### Задание 9
Убедитесь, что ручной расчёт совпадает с `model.predict()`. Это показывает, что нейронная сеть в данном примере не является «чёрным ящиком»: её вычисление можно воспроизвести по весам.


## 10. Небольшой эксперимент

Измените **один** параметр и повторите обучение:
- `epochs`: 20 вместо 200; **или**
- `learning_rate`: 0.005 вместо 0.05; **или**
- `batch_size`: 16 вместо 4.

Сравните итоговый loss, точность и график обучения с исходной моделью.


In [ ]:
# Ваш эксперимент

## Итоговый вывод
Кратко ответьте:
1. Что в этой задаче является входом и правильным ответом?
2. Что такое вес и смещение?
3. Для чего нужна функция активации sigmoid?
4. Что делает `fit()`?
5. Чем `loss` отличается от accuracy?
6. Как из двух выходных значений получить номер группы?
7. Что изменилось в вашем эксперименте?
